In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import DBSCAN
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDOneClassSVM
from sklearn.pipeline import make_pipeline

def run_all_4_models(X_scaled, combined, seed=42):
    """
    Shared helper: runs all 4 models on the given scaled feature matrix.
    Returns a dict of prediction arrays (1 = anomaly, 0 = normal).
    """
    results = {}

    # 1. Isolation Forest
    print("Running Isolation Forest...")
    if_model = IsolationForest(n_estimators=100, contamination=0.05, random_state=seed)
    results['if_pred'] = (if_model.fit_predict(X_scaled) == -1).astype(int)

    # 2. One-Class SVM
    print("Running One-Class SVM...")
    svm_pipeline = make_pipeline(
        Nystroem(kernel='rbf', gamma=None, n_components=500, random_state=seed),
        SGDOneClassSVM(nu=0.05, random_state=seed)
    )
    svm_pipeline.fit(X_scaled)
    results['svm_pred'] = (svm_pipeline.predict(X_scaled) == -1).astype(int)

    # 3. Autoencoder (seeded for reproducibility)
    print("Running Autoencoder...")
    tf.keras.utils.set_random_seed(seed)
    input_dim = X_scaled.shape[1]
    input_layer = keras.Input(shape=(input_dim,))
    enc = keras.layers.Dense(8, activation='relu')(input_layer)
    enc = keras.layers.Dense(4, activation='relu')(enc)
    dec = keras.layers.Dense(8, activation='relu')(enc)
    dec = keras.layers.Dense(input_dim, activation='linear')(dec)
    ae = keras.Model(input_layer, dec)
    ae.compile(optimizer='adam', loss='mse')
    ae.fit(X_scaled, X_scaled, epochs=20, batch_size=256, validation_split=0.1, verbose=0)
    reconstructed = ae.predict(X_scaled, verbose=0)
    mse = np.mean(np.power(X_scaled - reconstructed, 2), axis=1)
    ae_threshold = np.percentile(mse, 95)
    results['ae_pred'] = (mse > ae_threshold).astype(int)

    # 4. DBSCAN (stratified sample + KNN extension, matching Week 4 exactly)
    print("Running DBSCAN...")
    pca = PCA(n_components=5, random_state=seed)
    X_reduced = pca.fit_transform(X_scaled)

    combined_reset = combined.reset_index(drop=True)
    np.random.seed(seed)
    sample_idx = (
        combined_reset.groupby('user', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), 15), random_state=seed), include_groups=False)
        .index
    )
    remaining = list(set(range(len(combined_reset))) - set(sample_idx))
    extra_needed = max(0, 50000 - len(sample_idx))
    if extra_needed > 0:
        extra_idx = np.random.choice(remaining, size=extra_needed, replace=False)
        sample_idx = list(sample_idx) + list(extra_idx)

    X_sample = X_reduced[sample_idx]
    dbscan_model = DBSCAN(eps=0.8, min_samples=5, algorithm='ball_tree', n_jobs=-1)
    sample_labels = dbscan_model.fit_predict(X_sample)

    knn = KNeighborsClassifier(n_neighbors=3, algorithm='ball_tree', n_jobs=-1)
    knn.fit(X_sample, sample_labels)
    all_labels = knn.predict(X_reduced)
    results['dbscan_pred'] = (all_labels == -1).astype(int)

    return results


def evaluate_detection_full(feature_vectors, exfil_df, misuse_df, unauth_df, label="OBVIOUS"):
    """
    4-model evaluation (IF, SVM, AE, DBSCAN) against synthetic threat scenarios,
    using the same 2+/4 ensemble rule as the production Week 4 pipeline.
    """
    print(f"\n--- Evaluating Detection ({label} Threats, All 4 Models) ---")

    real_df = feature_vectors[FEATURE_COLS + ['user', 'day']].copy()
    real_df['is_threat'] = 0
    real_df['threat_type'] = 'normal'

    all_threats = pd.concat([exfil_df, misuse_df, unauth_df], ignore_index=True)
    combined = pd.concat([real_df, all_threats[real_df.columns]], ignore_index=True)

    print(f"Total records: {len(combined)} ({len(real_df)} normal, {len(all_threats)} threats)")

    X = combined[FEATURE_COLS].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    preds = run_all_4_models(X_scaled, combined)

    combined['if_pred'] = preds['if_pred']
    combined['svm_pred'] = preds['svm_pred']
    combined['ae_pred'] = preds['ae_pred']
    combined['dbscan_pred'] = preds['dbscan_pred']

    total_flags = preds['if_pred'] + preds['svm_pred'] + preds['ae_pred'] + preds['dbscan_pred']
    combined['ensemble_2plus'] = (total_flags >= 2).astype(int)  # matches Week 4 production rule
    combined['ensemble_3plus'] = (total_flags >= 3).astype(int)

    threat_types = all_threats['threat_type'].unique().tolist()
    print(f"\n=== {label} Detection Results by Threat Type (All 4 Models) ===")
    for threat_type in threat_types:
        subset = combined[combined['threat_type'] == threat_type]
        if len(subset) == 0:
            continue
        total = len(subset)
        print(f"\n{threat_type.upper()} ({total} scenarios):")
        print(f"  Isolation Forest:      {subset['if_pred'].sum()}/{total} ({subset['if_pred'].sum()/total*100:.1f}%)")
        print(f"  One-Class SVM:         {subset['svm_pred'].sum()}/{total} ({subset['svm_pred'].sum()/total*100:.1f}%)")
        print(f"  Autoencoder:           {subset['ae_pred'].sum()}/{total} ({subset['ae_pred'].sum()/total*100:.1f}%)")
        print(f"  DBSCAN:                {subset['dbscan_pred'].sum()}/{total} ({subset['dbscan_pred'].sum()/total*100:.1f}%)")
        print(f"  Ensemble (2+/4 agree): {subset['ensemble_2plus'].sum()}/{total} ({subset['ensemble_2plus'].sum()/total*100:.1f}%)")
        print(f"  Ensemble (3+/4 agree): {subset['ensemble_3plus'].sum()}/{total} ({subset['ensemble_3plus'].sum()/total*100:.1f}%)")

    return combined


# Obvious threats, all 4 models 
np.random.seed(42)
exfil_df    = simulate_exfiltration(feature_vectors)
misuse_df   = simulate_privilege_misuse(feature_vectors)
unauth_df   = simulate_unauthorized_access(feature_vectors, baseline_profiles)
combined_df_full = evaluate_detection_full(feature_vectors, exfil_df, misuse_df, unauth_df, label="OBVIOUS")

# Subtle threats, all 4 models 
np.random.seed(42)
subtle_exfil_df   = simulate_subtle_exfiltration(feature_vectors)
subtle_misuse_df  = simulate_subtle_privilege_misuse(feature_vectors)
subtle_unauth_df  = simulate_subtle_unauthorized_access(feature_vectors, baseline_profiles)
subtle_combined_df_full = evaluate_detection_full(feature_vectors, subtle_exfil_df, subtle_misuse_df, subtle_unauth_df, label="SUBTLE")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import DBSCAN
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import SGDOneClassSVM
from sklearn.pipeline import make_pipeline

def evaluate_detection(feature_vectors, exfil_df, misuse_df, unauth_df):
    """
    Evaluate detection on OBVIOUS threat scenarios using all 4 models:
    Isolation Forest, One-Class SVM, Autoencoder, DBSCAN.
    """
    print("\n--- Evaluating Detection (OBVIOUS Threats, All 4 Models) ---")

    real_df = feature_vectors[FEATURE_COLS + ['user', 'day']].copy()
    real_df['is_threat'] = 0
    real_df['threat_type'] = 'normal'

    all_threats = pd.concat([exfil_df, misuse_df, unauth_df], ignore_index=True)
    combined = pd.concat([real_df, all_threats[real_df.columns]], ignore_index=True)

    print(f"Total records: {len(combined)} ({len(real_df)} normal, {len(all_threats)} threats)")

    X = combined[FEATURE_COLS].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 1. Isolation Forest
    print("\nRunning Isolation Forest...")
    if_model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    if_preds = (if_model.fit_predict(X_scaled) == -1).astype(int)

    # 2. One-Class SVM
    print("Running One-Class SVM...")
    svm_pipeline = make_pipeline(
        Nystroem(kernel='rbf', gamma=None, n_components=500, random_state=42),
        SGDOneClassSVM(nu=0.05, random_state=42)
    )
    svm_pipeline.fit(X_scaled)
    svm_preds = (svm_pipeline.predict(X_scaled) == -1).astype(int)

    # 3. Autoencoder (seeded)
    print("Running Autoencoder...")
    tf.keras.utils.set_random_seed(42)
    input_dim = X_scaled.shape[1]
    input_layer = keras.Input(shape=(input_dim,))
    enc = keras.layers.Dense(8, activation='relu')(input_layer)
    enc = keras.layers.Dense(4, activation='relu')(enc)
    dec = keras.layers.Dense(8, activation='relu')(enc)
    dec = keras.layers.Dense(input_dim, activation='linear')(dec)
    ae = keras.Model(input_layer, dec)
    ae.compile(optimizer='adam', loss='mse')
    ae.fit(X_scaled, X_scaled, epochs=20, batch_size=256, validation_split=0.1, verbose=0)
    reconstructed = ae.predict(X_scaled, verbose=0)
    mse = np.mean(np.power(X_scaled - reconstructed, 2), axis=1)
    ae_threshold = np.percentile(mse, 95)
    ae_preds = (mse > ae_threshold).astype(int)

    # 4. DBSCAN (stratified sample + KNN extension, matching Week 4)
    print("Running DBSCAN...")
    pca = PCA(n_components=5, random_state=42)
    X_reduced = pca.fit_transform(X_scaled)

    combined_reset = combined.reset_index(drop=True)
    np.random.seed(42)
    sample_idx = (
        combined_reset.groupby('user', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), 15), random_state=42), include_groups=False)
        .index
    )
    remaining = list(set(range(len(combined_reset))) - set(sample_idx))
    extra_needed = max(0, 50000 - len(sample_idx))
    if extra_needed > 0:
        extra_idx = np.random.choice(remaining, size=extra_needed, replace=False)
        sample_idx = list(sample_idx) + list(extra_idx)

    X_sample = X_reduced[sample_idx]
    dbscan_model = DBSCAN(eps=0.8, min_samples=5, algorithm='ball_tree', n_jobs=-1)
    sample_labels = dbscan_model.fit_predict(X_sample)

    knn = KNeighborsClassifier(n_neighbors=3, algorithm='ball_tree', n_jobs=-1)
    knn.fit(X_sample, sample_labels)
    all_labels = knn.predict(X_reduced)
    dbscan_preds = (all_labels == -1).astype(int)

    # Ensembles (4 models)
    total_flags = if_preds + svm_preds + ae_preds + dbscan_preds
    ensemble_2plus = (total_flags >= 2).astype(int)  # matches Week 4 production rule
    ensemble_3plus = (total_flags >= 3).astype(int)

    combined['if_pred'] = if_preds
    combined['svm_pred'] = svm_preds
    combined['ae_pred'] = ae_preds
    combined['dbscan_pred'] = dbscan_preds
    combined['ensemble_2plus'] = ensemble_2plus
    combined['ensemble_3plus'] = ensemble_3plus

    print("\n=== OBVIOUS Detection Results by Threat Type (All 4 Models) ===")
    for threat_type in ['exfiltration', 'privilege_misuse', 'unauthorized_access']:
        subset = combined[combined['threat_type'] == threat_type]
        if len(subset) == 0:
            continue
        total = len(subset)
        print(f"\n{threat_type.upper()} ({total} scenarios):")
        print(f"  Isolation Forest:      {subset['if_pred'].sum()}/{total} ({subset['if_pred'].sum()/total*100:.1f}%)")
        print(f"  One-Class SVM:         {subset['svm_pred'].sum()}/{total} ({subset['svm_pred'].sum()/total*100:.1f}%)")
        print(f"  Autoencoder:           {subset['ae_pred'].sum()}/{total} ({subset['ae_pred'].sum()/total*100:.1f}%)")
        print(f"  DBSCAN:                {subset['dbscan_pred'].sum()}/{total} ({subset['dbscan_pred'].sum()/total*100:.1f}%)")
        print(f"  Ensemble (2+/4 agree): {subset['ensemble_2plus'].sum()}/{total} ({subset['ensemble_2plus'].sum()/total*100:.1f}%)")
        print(f"  Ensemble (3+/4 agree): {subset['ensemble_3plus'].sum()}/{total} ({subset['ensemble_3plus'].sum()/total*100:.1f}%)")

    return combined


# ── Run ──────────────────────────────────────────────────────────────────────
np.random.seed(42)

exfil_df    = simulate_exfiltration(feature_vectors)
misuse_df   = simulate_privilege_misuse(feature_vectors)
unauth_df   = simulate_unauthorized_access(feature_vectors, baseline_profiles)
combined_df = evaluate_detection(feature_vectors, exfil_df, misuse_df, unauth_df)

In [ ]:
def evaluate_subtle_detection(feature_vectors, subtle_exfil_df, subtle_misuse_df, subtle_unauth_df):
    """
    Evaluate detection on SUBTLE threat scenarios using all 4 models:
    Isolation Forest, One-Class SVM, Autoencoder, DBSCAN.
    """
    print("\n--- Evaluating Detection on SUBTLE Threats (All 4 Models) ---")

    real_df = feature_vectors[FEATURE_COLS + ['user', 'day']].copy()
    real_df['is_threat'] = 0
    real_df['threat_type'] = 'normal'

    all_subtle_threats = pd.concat([subtle_exfil_df, subtle_misuse_df, subtle_unauth_df], ignore_index=True)
    combined = pd.concat([real_df, all_subtle_threats[real_df.columns]], ignore_index=True)

    print(f"Total records: {len(combined)} ({len(real_df)} normal, {len(all_subtle_threats)} threats)")

    X = combined[FEATURE_COLS].fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 1. Isolation Forest
    print("\nRunning Isolation Forest...")
    if_model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
    if_preds = (if_model.fit_predict(X_scaled) == -1).astype(int)

    # 2. One-Class SVM
    print("Running One-Class SVM...")
    svm_pipeline = make_pipeline(
        Nystroem(kernel='rbf', gamma=None, n_components=500, random_state=42),
        SGDOneClassSVM(nu=0.05, random_state=42)
    )
    svm_pipeline.fit(X_scaled)
    svm_preds = (svm_pipeline.predict(X_scaled) == -1).astype(int)

    # 3. Autoencoder (seeded)
    print("Running Autoencoder...")
    tf.keras.utils.set_random_seed(42)
    input_dim = X_scaled.shape[1]
    input_layer = keras.Input(shape=(input_dim,))
    enc = keras.layers.Dense(8, activation='relu')(input_layer)
    enc = keras.layers.Dense(4, activation='relu')(enc)
    dec = keras.layers.Dense(8, activation='relu')(enc)
    dec = keras.layers.Dense(input_dim, activation='linear')(dec)
    ae = keras.Model(input_layer, dec)
    ae.compile(optimizer='adam', loss='mse')
    ae.fit(X_scaled, X_scaled, epochs=20, batch_size=256, validation_split=0.1, verbose=0)
    reconstructed = ae.predict(X_scaled, verbose=0)
    mse = np.mean(np.power(X_scaled - reconstructed, 2), axis=1)
    ae_threshold = np.percentile(mse, 95)
    ae_preds = (mse > ae_threshold).astype(int)

    # 4. DBSCAN (stratified sample + KNN extension, matching Week 4)
    print("Running DBSCAN...")
    pca = PCA(n_components=5, random_state=42)
    X_reduced = pca.fit_transform(X_scaled)

    combined_reset = combined.reset_index(drop=True)
    np.random.seed(42)
    sample_idx = (
        combined_reset.groupby('user', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), 15), random_state=42), include_groups=False)
        .index
    )
    remaining = list(set(range(len(combined_reset))) - set(sample_idx))
    extra_needed = max(0, 50000 - len(sample_idx))
    if extra_needed > 0:
        extra_idx = np.random.choice(remaining, size=extra_needed, replace=False)
        sample_idx = list(sample_idx) + list(extra_idx)

    X_sample = X_reduced[sample_idx]
    dbscan_model = DBSCAN(eps=0.8, min_samples=5, algorithm='ball_tree', n_jobs=-1)
    sample_labels = dbscan_model.fit_predict(X_sample)

    knn = KNeighborsClassifier(n_neighbors=3, algorithm='ball_tree', n_jobs=-1)
    knn.fit(X_sample, sample_labels)
    all_labels = knn.predict(X_reduced)
    dbscan_preds = (all_labels == -1).astype(int)

    # Ensembles (4 models)
    total_flags = if_preds + svm_preds + ae_preds + dbscan_preds
    ensemble_2plus = (total_flags >= 2).astype(int)
    ensemble_3plus = (total_flags >= 3).astype(int)

    combined['if_pred'] = if_preds
    combined['svm_pred'] = svm_preds
    combined['ae_pred'] = ae_preds
    combined['dbscan_pred'] = dbscan_preds
    combined['ensemble_2plus'] = ensemble_2plus
    combined['ensemble_3plus'] = ensemble_3plus

    print("\n=== SUBTLE Detection Results by Threat Type (All 4 Models) ===")
    for threat_type in ['subtle_exfiltration', 'subtle_privilege_misuse', 'subtle_unauthorized_access']:
        subset = combined[combined['threat_type'] == threat_type]
        if len(subset) == 0:
            continue
        total = len(subset)
        print(f"\n{threat_type.upper()} ({total} scenarios):")
        print(f"  Isolation Forest:      {subset['if_pred'].sum()}/{total} ({subset['if_pred'].sum()/total*100:.1f}%)")
        print(f"  One-Class SVM:         {subset['svm_pred'].sum()}/{total} ({subset['svm_pred'].sum()/total*100:.1f}%)")
        print(f"  Autoencoder:           {subset['ae_pred'].sum()}/{total} ({subset['ae_pred'].sum()/total*100:.1f}%)")
        print(f"  DBSCAN:                {subset['dbscan_pred'].sum()}/{total} ({subset['dbscan_pred'].sum()/total*100:.1f}%)")
        print(f"  Ensemble (2+/4 agree): {subset['ensemble_2plus'].sum()}/{total} ({subset['ensemble_2plus'].sum()/total*100:.1f}%)")
        print(f"  Ensemble (3+/4 agree): {subset['ensemble_3plus'].sum()}/{total} ({subset['ensemble_3plus'].sum()/total*100:.1f}%)")

    return combined


# ── Run ──────────────────────────────────────────────────────────────────────
np.random.seed(42)

subtle_exfil_df   = simulate_subtle_exfiltration(feature_vectors)
subtle_misuse_df  = simulate_subtle_privilege_misuse(feature_vectors)
subtle_unauth_df  = simulate_subtle_unauthorized_access(feature_vectors, baseline_profiles)
subtle_combined_df = evaluate_subtle_detection(feature_vectors, subtle_exfil_df, subtle_misuse_df, subtle_unauth_df)